<h1><center>Laboratorio 3: La desperación de Mr. Cheems 🐼</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo:

- Nombre de alumno 1: Naomí Cautivo B.
- Nombre de alumno 2: Máximo Flores Valenzuela


### **Link de repositorio de GitHub:** [maxfloresv/MDS7202](https://github.com/maxfloresv/MDS7202/)

## Temas a tratar
- Aplicar Pandas para obtener características de un DataFrame.
- Aplicar Pipelines y Column Transformers

## Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### Objetivos principales del laboratorio
- Comprender cómo aplicar pipelines de Scikit-Learn para generar procesos más limpios en Feature Engineering.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `numpy`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre arreglos (*o tensores*).

## Descripción del laboratorio.

### Importamos librerias utiles 😸

In [ ]:
import numpy as np
import pandas as pd
import datetime
from IPython.display import HTML

!pip install --upgrade plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer 
from sklearn.preprocessing import FunctionTransformer

In [ ]:
# En este laboratorio, no se usará Google Colab. Si se desea usar, se debe descomentar.
"""
try:
    from google.colab import drive
    drive.mount("/content/drive")
    path = 'Dirección donde tiene los archivos en el Drive'
except:
    print('Ignorando conexión drive-colab')
"""

# Feature engineering en datos de retail 🛍️

### 0. Cargar Dataset

<p align="center">
  <img width=300 src="https://s1.eestatic.com/2018/04/14/social/la_jungla_-_social_299733421_73842361_854x640.jpg">
</p>

Mr. Cheems, gerente de una cotizada tienda de retail en Europa, les solicita si pueden analizar los datos de algunas de sus tiendas. En una reunión, Mr Cheems le comenta que la calidad de sus datos no es muy buena, por lo que le solicita a usted que limpie su base de datos y cree nuevos atributos relevantes para el negocio.

Por ello, el área de ventas les entrega archivo llamado `online_retail_data.pickle` el cual usted decide cargar a continuación.

In [ ]:
df_retail = pd.read_pickle('online_retail_data.pickle')
df_retail.head()

In [ ]:
df_retail.info()

### 1. Función para explorar características [0.5 puntos]

<p align="center">
  <img width=300 src="https://editor.analyticsvidhya.com/uploads/47389meme.png">
</p>




Tras inspeccionar brevemente los datos proporcionados, usted decide crear una función que realice lo siguiente:
- Plotee un histograma para las variables precios y cantidad. [0.3 puntos]
- Imprima un conteo de datos nulos por variable [0.2 puntos]

**Nota**: Para generar los gráficos no es obligatorio el uso de `plotly`, pero si es altamente recomendado. Pueden encontrar más información de esta librería en este [enlace](https://plotly.com/python/).

**Respuesta:**

In [ ]:
def explore_data(dataframe_in) -> None:
  """
  Grafica histogramas para las variables «Price» y «Quantity».
  Imprime en pantalla un conteo de datos nulos por variable.

  Parameters
  ----------
  dataframe_in : pd.DataFrame
    DataFrame a explorar.
  """
  fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Histograma de la variable Price", 
    "Histograma de la variable Quantity"
  ))

  fig.add_trace(go.Histogram(x=dataframe_in["Price"], name="Price", nbinsx=25), row=1, col=1)
  fig.add_trace(go.Histogram(x=dataframe_in["Quantity"], name="Quantity", nbinsx=25), row=1, col=2)

  fig.show()

  print("Conteo de datos nulos por variable:")
  print(dataframe_in.isnull().sum())

### 2. Eliminando outliers [1.0 puntos]

<p align="center">
  <img width=300 src="https://media.licdn.com/dms/image/C5612AQGdXKCka7HumA/article-cover_image-shrink_600_2000/0/1520056407281?e=2147483647&v=beta&t=VZcfjjzjK4LxXdZkSu1KisWC0Ry8bk4tPCn3R8aYdNM">
</p>




#### 2.1 Creando la clase IQR [0.5 puntos]

Entre las falencias de los datos, Mr. Cheems le comenta que a veces los operadores no ingresan el precio correcto de los productos. Mr. Cheems le comenta que se dio cuenta de este fenómeno porque hay productos con precios exagerádamente altos o bajos. Por lo cual usted decide eliminar outliers del dataframe a traves del rango intercuartil el cual cuenta con los siguientes pasos:

1. Calcular el primer cuartil $Q1$ y el tercer cuartil $Q3$. Hint: utilice el método `quantile()`

2. Calcular el rango intercuartil (RIC): $RIC = Q3 - Q1$

3. Calcular los límites para identificar outliers:
 - Límite inferior: $~~Q1 - \lambda \cdot RIC$
 - Límite superior: $~~Q3 + \lambda \cdot RIC$

4. Eliminar outliers: Los outliers son los datos que están por debajo del límite inferior o por encima del límite superior.


Para realizar dicha tarea, usted decide crear una clase llamada `IQR()` utilizando `BaseEstimator` y `TransformerMixin` para realizar una transformación de cada una de las columnas numéricas del DataFrame utilizando `ColumnTransformer()` más tarde. Considere que lambda debe ser $\lambda$ un parámetro a definir por el usuario.

**Hint:** tome como referencia el siguiente [enlace](https://sklearn-template.readthedocs.io/en/latest/user_guide.html#transformer).

**Nota:** No modificar el método set_output de la clase IQR

**Respuesta:**

In [ ]:
class IQR(BaseEstimator, TransformerMixin):
  """
  Implementa el método de detección y tratamiento de outliers basado 
  en el rango intercuartil (IQR).
  """
  def __init__(self, lam):
    self.lam = lam

  def fit(self, X: pd.DataFrame):
    """
    Aprende los límites inferior y superior para la detección de outliers.

    Parameters
    ----------
    X : pd.DataFrame
      Conjunto de datos de entrada.
    """
    self.q1 = X.quantile(0.25)
    self.q3 = X.quantile(0.75)
    self.iqr = self.q3 - self.q1
    self.lower_bound = self.q1 - self.lam * self.iqr
    self.upper_bound = self.q3 + self.lam * self.iqr
    return self

  def transform(self, X: pd.DataFrame):
    """
    Aplica la transformación a los datos de entrada, eliminando los outliers.

    Parameters
    ----------
    X : pd.DataFrame
      Conjunto de datos de entrada.

    Returns
    -------
    pd.DataFrame
      Conjunto de datos transformado.
    """
    X = X.copy()
    return X.clip(lower=self.lower_bound, upper=self.upper_bound, axis=1)

  def set_output(self, transform='default'):
    return self

#### 2.2 Creación del Pipeline [0.5 puntos]

Para comenzar introduciéndose en el uso de pipeline, usted decide definir un pipeline con el Transformer previamente definido. Además, usted decide visualizar cómo cambia la distribución de las variables Precio y Cantidad antes y despues de aplicar IQR. Para ello, usted aplica los siguientes pasos:

- Definir un pipeline llamado `numeric_transformations` para las variables precio y cantidad con la transformación IQR. [0.1 puntos]
- Defina un column transformer que aplique `numeric_transformations` para las variables numéricas y `passthrough` para las variables categóricas. Adicionalmente, fije el parámetro `verbose_feature_names_out` en `False`. Ver hint al final [0.1 puntos]
- Defina el dataframe `df_iqr` aplicado el column transformer a los datos proporcionados por Mr. Cheems considerando un valor de $\lambda$ que tenga un desempeño aceptable para ambas variables. [0.1 puntos]
- Usar `explore_data` en `df_retail` y en `df_iqr`.  [0.1 puntos]
- Reportar los cambios observados en la distribución de las variables. ¿Qué sucede al aumentar el valor de lambda? [0.1 puntos]


**Hint:** El transformador `passthrough` está predefinido y es una opción que puedes usar para las columnas que no deseas transformar. Al especificar 'passthrough' para una parte de tu ColumnTransformer, las columnas correspondientes pasarán a través del ColumnTransformer sin ninguna modificación. El siguiente [enlace](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) le puede ser útil.

**Nota:** Mantenga el método set_output del column transformer con la transformación `pandas` para obtener un dataframe una vez aplicado el column transformer.

**Respuesta:**

Apóyese de la siguiente estructura para su respuesta:

In [ ]:
numerical_columns = ["Quantity", "Price"]
categorical_columns = list(
  filter(
    lambda col: col not in numerical_columns, 
    df_retail.columns
  )
)

numeric_transformations = Pipeline([('iqr_outliers', IQR(lam=1.5))])
column_transformer = ColumnTransformer([('numerical', numeric_transformations, numerical_columns),
                                        ('categorical', 'passthrough', categorical_columns)
                                        ],
                                        verbose_feature_names_out=False)

column_transformer.set_output(transform='pandas')
df_iqr = column_transformer.fit_transform(df_retail)

explore_data(df_retail)
explore_data(df_iqr)

> **Respuesta**: Dado que estamos recortando _outliers_, la aplicación del `ColumnTransformer` permite ver mejor la distribución de los datos de cada variable, dado que la escala tiende a estar concentrada en un intervalo más pequeño. Al aumentar el valor de $\lambda$, ocurre que el intervalo del eje $X$ se hace más grande. Esto tiene sentido, porque se están considerando más datos al aumentar la tolerancia.

### 3. Agregando un imputer al pipeline [1.0 puntos]



<p align="center">
  <img width=300 src="https://media.makeameme.org/created/hmm-there-is.jpg">
</p>

Para continuar con la limpieza del dataframe usted decide imputar los datos nulos de las variables numéricas, para lo cual decide realizar las siguientes tareas:

1. Crear un pipeline para variables categóricas llamado `categoric_transformations` con un paso llamado `mode_imputer`, en el cual se imputen los datos faltantes por la categoría más frecuente.
2. Agregar al pipeline `numeric_transformations` un paso llamado `mean_imputer`, en el cual se imputen los datos por la media usando [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) [0.1 puntos]
3. Crear y aplicar un `ColumnTransformer` actualizado con los pipelines `categoric_transformations` y `numeric_transformations` a `df_retail`, creando un dataframe llamado `df_mean_imputer`. [0.1 puntos]
4. Comparar los resultados de `explore_data` en `df_mean_imputer` y `df_iqr`. ¿Qué diferencias observa en la distribución de los datos? [0.2 puntos]
5. Cambiar el imputer de `numeric_transformations` por [KNNImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html) y definir un nuevo dataframe llamado `df_knn_imputer`, aplicando el nuevo ColumnTransformer a `df_retail`. En caso de los tiempos de ejecución sean altos puede probar a reducir el parámetro `n_neighbors`. [0.1 puntos]
6. Comparar los resultados de `explore_data` en `df_knn_imputer` y `df_iqr`. ¿Qué diferencias observa en la distribución de los datos? [0.2 puntos]
7. Comparar los resultados de `explore_data` en `df_knn_imputer` y `df_mean_imputer`. ¿Cuál método de imputación es mejor? Deje el método escogido en el ColumnTransformer. [0.2 puntos]

**Nota: Fije el parámetro verbose_feature_names_out en `False` y utilice el método set_output con transformación `pandas` en cada ColumnTransformer para obtener como salida un dataframe.**

**Respuesta:**

In [ ]:
# 1. a 4. Aplicando SimpleImputer
categoric_transformations = Pipeline([
  ('mode_imputer', FunctionTransformer(lambda X: X.fillna(X.mode().iloc[0]), validate=False))
])
numeric_transformations = Pipeline([
  ('iqr_outliers', IQR(lam=1.5)),
  ('mean_imputer', SimpleImputer(strategy='mean')),
])

ct = ColumnTransformer([('numerical', numeric_transformations, numerical_columns),
                        ('categorical', categoric_transformations, categorical_columns)
                        ],
                        verbose_feature_names_out=False)

ct.set_output(transform='pandas')
df_mean_imputer = ct.fit_transform(df_retail)

print("[1. a 4.] Análisis para df_iqr")
explore_data(df_iqr)

print("[1. a 4.] Análisis para df_mean_imputer")
explore_data(df_mean_imputer)

# 5. a 7. Aplicando KNNImputer
numeric_transformations = Pipeline([
  ('iqr_outliers', IQR(lam=1.5)),
  ('knn_imputer', KNNImputer(n_neighbors=3)),
])

ct_knn = ColumnTransformer([('numerical', numeric_transformations, numerical_columns),
                            ('categorical', categoric_transformations, categorical_columns)
                           ],
                           verbose_feature_names_out=False)

ct_knn.set_output(transform='pandas')
df_knn_imputer = ct_knn.fit_transform(df_retail)

print("[5. a 7.] Análisis para df_iqr")
explore_data(df_iqr)

print("[5. a 7.] Análisis para df_knn_imputer")
explore_data(df_knn_imputer)

print("[5. a 7.] Análisis para df_mean_imputer")
explore_data(df_mean_imputer)

- **(4.)** Comparar los resultados de `explore_data` en `df_mean_imputer` y `df_iqr`. ¿Qué diferencias observa en la distribución de los datos?
> **Respuesta**: `df_mean_imputer` aumenta la frecuencia en la barra del histograma que representa al intervalo $[2.5, 2.99)$ para la variable «Price». Por otro lado, en la variable «Quantity» aumenta la frecuencia en la barra que representa al intervalo $[8, 9)$. Esto tiene sentido, pues los valores nulos ($\sim 8.000$) son reemplazados por la media después de haber aplicado _clip_ sobre los _outliers_ con la clase `IQR`.
- **(6.)** Comparar los resultados de `explore_data` en `df_knn_imputer` y `df_iqr`. ¿Qué diferencias observa en la distribución de los datos?
> **Respuesta**: `df_knn_imputer` mantiene una distribución similar de los datos con respecto a `df_iqr`, sin embargo, la frecuencia de cada _bin_ es levemente mayor con una proporción de aumento parecida. Por ejemplo, se puede observar que el rango $[0, 1.9)$ para la variable «Quantity» aumenta en aproximadamente $1.100$ observaciones, fenómeno que es similar en el rango $[2, 3)$, donde aumenta en $\approx 1.200$. El aumento de observaciones en cada intervalo se justifica con la imputación, y que las proporciones de aumento sean parecidas se justifica dado que ya no se elige un valor fijo para imputar, sino que el cálculo se basa en distancias.
- **(7.)** Comparar los resultados de `explore_data` en `df_knn_imputer` y `df_mean_imputer`. ¿Cuál método de imputación es mejor?
> **Respuesta**: Siguiendo lo que se ha dicho antes, la diferencia principal es que `df_mean_imputer` modifica notoriamente la distribución, pues hay un _bin_ que es mucho más alto por gráfico. En el caso del histograma de la variable «Price», es el _bin_ asociado al intervalo $[2.5, 2.99)$, y en el caso de «Quantity», $[8, 9)$. Con respecto a la versión donde los _outliers_ fuera de un rango fueron eliminados (`df_knn`), el _imputer_ que mejor conserva la distribución es `df_knn_imputer`, pues si bien aumenta la frecuencia en cada _bin_, lo hace de manera pareja para todos. Si consideramos que no queremos sesgar la forma de la distribución, el mejor _imputer_ es `df_knn_imputer` para este caso. Así, a partir de ahora en adelante, se ocupará el _transformer_ `ct_knn`.

### 4. Creación de nuevas features [2.0 puntos]

<p align="center">
  <img width=250 src="https://miro.medium.com/max/1000/1*JtTWgAcfVTWV8OTjT47Atg.jpeg">
</p>


#### 4.1 Definicion de LRMFP [1.0 puntos]

Dado que Mr. Lepin está interesado en obtener nuevos atributos relevantes para su negocio, su equipo de expertos sugiere la construcción de variables **LRMFP**, las que se construyen en base a las siguientes definiciones:

- **Length (L)**: Intervalo de tiempo, en días, entre la primera y la última visita del cliente. Mientras mas grande sea el valor, mas fiel es el cliente.

- **Recency (R)**: Indica hace cuanto tiempo el cliente realizo su ultima compra. Notar que para este caso, mientras mas grande es el valor, menos interes posee el usuario para repetir una compra en uno de los locales. **Considere "hoy" como la fecha mas reciente del dataset**.

- **Monetary (M)**: El término "monetario" se refiere a la cantidad media de dinero gastada por cada visita del cliente durante el período de observación y refleja la contribución del cliente a los ingresos de la empresa.

- **Frequency (F)**: Se refiere al número total de visitas del cliente durante el periodo de observación. Cuanto mayor sea la frecuencia, mayor será la fidelidad del cliente.

- **Periodicity (P)**: Representa si los clientes visitan las tiendas con regularidad.

$$Periodicity(n)=std(IVT_1, ..., IVT_n)$$

Donde $IVT$ denota el tiempo entre visitas y n representa el número de valores de tiempo entre visitas de un cliente.


$$IVT_i=date\_diff(t_{i+1},t)$$

En base a las definiciones señaladas, diseñe una función que permita obtener las características **LRMFP** recibiendo un DataFrame como entrada. Para esto, no estará permitido el uso de iteradores, utilice todas las herramientas que les ofrece `pandas` para realizar esto.

Una referencia que le puede ser útil es el [documento original](https://www.researchgate.net/publication/315979555_LRFMP_model_for_customer_segmentation_in_the_grocery_retail_industry_a_case_study) en donde se propone este método.

**<u>Formato</u> del Resultado Esperado:**

| Customer ID | Length | Recency | Frequency | Monetary | Periodicity |
|------------:|-------:|--------:|----------:|---------:|------------:|
|   12346.0   |    294 |      67 |        46 |   -64.68 |        37.0 |
|   12347.0   |     37 |       3 |        71 |  1323.32 |         0.0 |
|   12349.0   |    327 |      43 |       107 |  2646.99 |        78.0 |
|   12352.0   |     16 |      11 |        18 |   343.80 |         0.0 |
|   12356.0   |     44 |      16 |        84 |  3562.25 |        12.0 |

**Respuesta:**

In [ ]:
def custom_features(dataframe_in) -> pd.DataFrame:
  """
  Crea las características solicitadas:  
  - Length: Cuánto tiempo ha pasado entre la primera y última compra del cliente.
  - Recency: Cuánto tiempo ha pasado desde la última compra del cliente hasta la fecha.
  - Monetary: Cuánto gasta en promedio el cliente por compra.
  - Frequency: Cuántas compras ha realizado el cliente.
  - Periodicity: Periodicidad de compra del cliente (mediante la desviación estándar de los intervalos entre compras).

  Parameters
  ----------
  dataframe_in : pd.DataFrame
    Conjunto de datos de entrada.

  Returns
  -------
  pd.DataFrame
    Conjunto de datos transformado.
  """
  df_features = dataframe_in.copy()
  # observed=True permite desprenderse de los valores nulos
  grouped_by = df_features.groupby('Customer ID', observed=True)
  length = (grouped_by['InvoiceDate'].max() - grouped_by['InvoiceDate'].min()).dt.days
  recency = (df_features['InvoiceDate'].max() - grouped_by['InvoiceDate'].max()).dt.days
  # Se considera la interacción Quantity * Price para ver los gastos totales por compra
  monetary = (df_features['Quantity'] * df_features['Price']).groupby(df_features['Customer ID']).mean()
  # Pueden haber varias compras asociadas a una factura, pero sólo visitó una vez la tienda
  frequency = grouped_by['Invoice'].nunique()
  periodicity = grouped_by['InvoiceDate'].apply(lambda x: x.sort_values().diff().dt.days.std())
  df_features = pd.DataFrame({
    'Length': length,
    'Recency': recency,
    'Monetary': monetary,
    'Frequency': frequency,
    'Periodicity': periodicity
  })
  df_features.reset_index(inplace=True)
  return df_features

In [ ]:
custom_features(df_knn_imputer)

#### 4.2 Agregando las custom features [1.0 puntos]

Ahora, usted decide agregar al pipeline las nuevas variables creadas, para lo cual realiza las siguientes tareas:

1. Cree un nuevo pipeline llamado `retail_pipeline` que encapsule el ColumnTransformer y calcule las LRMFP. El primer paso del pipeline llámelo  `col_tranformer` y el segundo paso llámelo `custom_features`, incorpora las nuevas variables al dataframe. Hint: les puede ser útil investigar [este](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html) método. [0.1 puntos]
2. Aplicar el pipeline actualizado a los datos proporcionados por Mr. Cheems, creando un nuevo dataframe llamado `df_custom`. [0.1 puntos]
3. Explorar la distribución de las nuevas variables con `explore_data` y comentar brevemente (2-3 líneas) características de cada custom feature. [0.5 puntos]
5. Entregar un insight para el negocio en base a las nuevas variables. [0.3 puntos]

**Nota:** Recuerde fijar el parámetro `verbose_feature_names_out` en `False` e incorporar el método `set_output` para obtener una salida en formato dataframe del ColumnTransformer.

**Respuesta**

In [ ]:
feature_transformer = FunctionTransformer(custom_features, validate=False)
retail_pipeline = Pipeline([
    ('col_transformer', ct_knn),
    ('custom_features', feature_transformer)
])

retail_pipeline.set_output(transform='pandas')
df_custom = retail_pipeline.fit_transform(df_retail)

Para explorar la distribución de las nuevas variables, crearemos la función `explore_data_custom`:

In [ ]:
def explore_data_custom(dataframe_in) -> None:
  """
  Grafica histogramas para las variables nuevas LRMFP.
  Imprime en pantalla un conteo de datos nulos por cada variable nueva.

  Parameters
  ----------
  dataframe_in : pd.DataFrame
    DataFrame a explorar.
  """
  fig = make_subplots(rows=3, cols=2, subplot_titles=(
    "Histograma de la variable Length",
    "Histograma de la variable Recency",
    "Histograma de la variable Monetary",
    "Histograma de la variable Frequency",
    "Histograma de la variable Periodicity"
  ))

  fig.add_trace(go.Histogram(x=dataframe_in["Length"], name="Length", nbinsx=25), row=1, col=1)
  fig.add_trace(go.Histogram(x=dataframe_in["Recency"], name="Recency", nbinsx=25), row=1, col=2)
  fig.add_trace(go.Histogram(x=dataframe_in["Monetary"], name="Monetary", nbinsx=25), row=2, col=1)
  fig.add_trace(go.Histogram(x=dataframe_in["Frequency"], name="Frequency", nbinsx=25), row=2, col=2)
  fig.add_trace(go.Histogram(x=dataframe_in["Periodicity"], name="Periodicity", nbinsx=25), row=3, col=1)

  fig.show()

  print("Conteo de datos nulos por variable:")
  print(dataframe_in[
    ["Length", "Recency", "Monetary", "Frequency", "Periodicity"]
  ].isnull().sum())

A continuación, la llamaremos con el _Data Frame_ `df_custom`:

In [ ]:
explore_data_custom(df_custom)

- Explorar la distribución de las nuevas variables con `explore_data` y comentar brevemente (2-3 líneas) características de cada custom feature.

> **Respuesta**: Sobre esto, podemos decir lo siguiente:
> 1. _Length_ (en días): La mayoría de los clientes se concentra en el intervalo $[0, 19)$. Luego, hay una cola larga hasta el intervalo $[200, 300]$, donde vuelve a aumentar la cantidad de clientes. La concentración del aumento en este último rango está en el intervalo $[260, 279)$. Esto indica el tiempo que ha pasado entre la primera y última compra del cliente.
> 2. _Recency_ (en días): En esta distribución, es mucho más notoria la concentración en el intervalo $[0, 100]$, dado que la cola larga no modifica su comportamiento hasta los valores extremos $>300$. Esto indica qué tan recientes son las compras de los clientes, considerando como referencia de hoy la fecha más reciente del _dataset_.
> 3. _Monetary_ (en euros): La mayoría de los clientes se concentra en el intervalo $[0, 50]$, y los valores $>50$ pueden ser considerados extremos dado lo infrecuentes que son. Esto revela información sobre cuánto dinero gastan los clientes en promedio.
> 4. _Frequency_ (unidades): La concentración de los clientes está en el intervalo $[0, 9)$, quizás agregando las observaciones que hay en $[10, 19)$, que se ven notoriamente superiores en frecuencia a las que tienen valores $\ge 19$. Así, podríamos considerar todo lo que esté en el intervalo $\ge 19$ como _outliers_. Esto revela información sobre cuánto compran los clientes en total.
> 5. _Periodicity_ (en días): La concentración de los clientes está en el intervalo $[0, 50)$, considerando que hay un comportamiento estrictamente decreciente en el subintervalo $[15, 50) \subset [0, 50)$, que se extiende hacia los valores atípicos $\ge 50$. Esto representa una medida de si hay o no una periodicidad en las compras de los clientes.

- Entregar un insight para el negocio en base a las nuevas variables.

> **Respuesta**: _Length_ nos sugiere que la mayoría de los usuarios **compra frecuentemente**, dado que se demoran entre $0$ a $19$ días en repetir una compra. Sin embargo, hay aproximadamente $> 2.000$ clientes que no compran hace mínimo $200$ días. Por otro lado, _Recency_ nos sugiere que hay **alta retención** (o un alto volumen de clientes nuevos), pues la mayoría de los clientes compraron hace a lo más $50$ días, siendo la moda entre $0$ a $19$ días desde la fecha de referencia. _Monetary_ nos indica que muy pocos clientes exceden en promedio de los $50$ euros en sus compras; la mayoría realiza compras de $15$ a $25$ euros. _Frequency_ revela que no hay muchos clientes que compran más de $10$ veces, entonces podemos decir que la fidelidad en general es baja. Hay clientes que han comprado en más de $200$ oportunidades distintas, así que no se puede concluir que la tienda lleva poco tiempo abierta. Por último, _Periodicity_ nos indica que no hay una periodicidad clara en las compras de los clientes, es decir, no podemos «predecir» sin tanto error cuándo serán sus próximas compras dada una referencia. Si bien, hay $1.121$ observaciones en el rango $[0, 5)$, la gran mayoría está en los _bins_ que representan intervalos $\ge 5$.


### 5. MinMax Scaler [1.0 puntos]

<p align="center">
  <img width=300 src="https://i.imgflip.com/1fsprn.jpg">
</p>


#### 5.1 Definición del Column Transformer [0.5 puntos]

Construya una clase llamada `MinMax()` para realizar una transformación de cada una de las columnas de un DataFrame utilizando `ColumnTransformer()`. Recuerde  usar `BaseEstimator` y `TransformerMixin`.


 Para esto considere que Min-Max escaler queda dada por la ecuación:

$$MinMax = \dfrac{x-min(x)}{max(x) - min(x)}$$


Consulte el siguiente [link](https://sklearn-template.readthedocs.io/en/latest/user_guide.html#transformer) si tiene dudas sobre la creación de custom transformers.

**Respuesta:**

In [ ]:
class MinMax(BaseEstimator, TransformerMixin):
    def fit(self, X):
        """
        Aprende los valores mínimo y máximo para la normalización Min-Max.

        Parameters
        ----------
        X : pd.DataFrame
          Conjunto de datos de entrenamiento.
        """
        self.x_min = X.min()
        self.x_max = X.max()
        return self

    def transform(self, X):
        """
        Aplica una normalización Min-Max sobre los datos de entrada.

        Parameters
        ----------
        X : pd.DataFrame
            Conjunto de datos a transformar.
        """
        return (X - self.x_min) / (self.x_max - self.x_min)

    def set_output(self, transform='default'):
        return self

#### 5.2 Incorporando MinMax al pipeline [0.5 puntos]

Ahora, usted decide agregar el escalamiento al pipeline, para lo que decide seguir los siguientes pasos:

- Agregar el paso `minmax` al pipeline `numeric_transformations`, haciendo uso de la clase creada. [0.1 puntos]
- Defina el dataframe `df_minmax` aplicando el ColumnTransformer actualizado a los datos proporcionados por Mr. Cheems. [0.1 puntos]
- Usar `explore_data` en `df_retail` y en `df_minmax`. [0.1 puntos]
- Reportar los cambios observados en la distribución de las variables.  [0.2 puntos]

**Nota:** Recuerde fijar el parámetro `verbose_feature_names_out` en `False` e incorporar el método `set_output` para obtener una salida en formato dataframe del ColumnTransformer.

**Respuesta:**

In [ ]:
# El minmax se aplica al final, cuando ya tenemos la imputación
numeric_transformations = Pipeline([
  ('iqr_outliers', IQR(lam=1.5)),
  ('knn_imputer', KNNImputer(n_neighbors=3)),
  ('minmax', MinMax())
])

ct_minmax = ColumnTransformer([('numerical', numeric_transformations, numerical_columns),
                               ('categorical', categoric_transformations, categorical_columns)
                              ],
                              verbose_feature_names_out=False)

ct_minmax.set_output(transform='pandas')
df_minmax = ct_minmax.fit_transform(df_retail)

print("Análisis para df_retail")
explore_data(df_retail)

print("Análisis para df_minmax")
explore_data(df_minmax)

- Reportar los cambios observados en la distribución de las variables.

> **Respuesta**: Hay cambios muy grandes en la distribución de las variables. Primero que todo, ya no hay una distorsión de la escala debido a los _outliers_, dado que la aplicación de `clip(...)` mediante `IQR(...)` reemplaza los valores extremos. Después, podemos ver notoriamente el comportamiento local de la distribución, dado que `KNNImputer(...)` no la modifica significativamente como se vio _a priori_. Lo último, es que los valores están encerrados en el intervalo $[0, 1]$ por la aplicación de `MinMax(...)` (ver eje $X$), entonces son más comparables entre sí.

### 6. Pregunta teórica [0.5 puntos]

<p align="center">
  <img width=300 src="https://file.coinexstatic.com/2023-09-19/166BAC031F222E5910954E7D7D0BC844.png">
</p>

Finalmente, explíquele a Mr. Cheems porqué es útil la creación de pipelines al momento de hacer Feature Engineering en Machine Learning.

> **Respuesta**: La creación de _pipelines_ es importante a la hora de hacer _Feature Engineering_ por varios motivos:
> 1. _Permite generar un flujo reproducible, con pasos claros, y referencias a las clases que transforman los datos._ Aplicado al laboratorio, se nos hizo mucho más fácil saber cuándo estábamos aplicando una serie de pasos en ejecuciones distintas de _Column Transformers_. Como el _pipeline_ ya tiene la información de las transformaciones a aplicar, no puede ocurrir el escenario donde se nos olvide aplicar alguna, como sí lo es cuando todas las responsabilidades estén en entidades separadas (y no en la entidad `Pipeline`).
> 2. _Permite ajustar fácilmente los hiperparámetros del modelo_. En el laboratorio, cuando cambiábamos los hiperparámetros (p. ej., `n_neighbors` para `KNNImputer`), el hecho de que todo estuviese encapsulado en una entidad `Pipeline` permitió saber con seguridad qué parte del código debíamos atacar para realizar cambios en ellos.
> 3. _Permite depurar más rápido los errores_. En el laboratorio, cada _pipeline_ se definió a partir de una serie de pasos como se mencionó en el punto anterior. Cuando existían errores, dado que la secuencia de aplicación de transformaciones era clara, fue mucho más fácil identificar la fuente de errores.
> 4. _Evita problemas de Data Leakage_. Como se sigue estrictamente la estructura de Scikit-Learn, la responsabilidad del método `fit(...)` es netamente calcular parámetros que serán usados en `transform(...)` a partir de los datos de entrenamiento. En `transform(...)`, se aplican las transformaciones sobre el conjunto de _testing_, a partir de lo aprendido en `fit(...)`.
>
> En general, se nos hizo mucho más fácil el seguimiento sobre el flujo _input_ $\to$ _output_.